In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, HfArgumentParser
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

# dataset = load_dataset("timdettmers/openassistant-guanaco", split="train")

parser, = HfArgumentParser((SFTConfig,))
args = parser.parse_args_into_dataclasses()
model_name_or_path = "/home1/09636/zyliu/scratch/base_models/deepseek/hf/deepseek-coder-1.3b-base"
model = AutoModelForCausalLM.from_pretrained(model_name_or_path)
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
tokenizer.sep_token = tokenizer.cls_token = tokenizer.mask_token = tokenizer.pad_token

model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id


/work/09636/zyliu/vista/miniconda3/envs/astro/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TypeError: cannot unpack non-iterable HfArgumentParser object

In [2]:
train_dataset = load_dataset("json", data_files="/home1/09636/zyliu/work/OLMo/olmo_data/astro_sft/astro_sft_train.jsonl", split="train")
valid_dataset = load_dataset("json", data_files="/home1/09636/zyliu/work/OLMo/olmo_data/astro_sft/astro_sft_valid.jsonl", split="train")

In [3]:
response_template = "### Response:\n"

In [5]:
model.config.max_position_embeddings

16384

In [ ]:
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

lr=args.learning_rate
wd=args.weight_decay
warmup_ratio=args.warmup_ratio
lr_scheduler_type="linear"
seed=42

gradient_accumulation_steps = 2
max_sequence_length = 2048

training_args = SFTConfig(
    output_dir="/home1/09636/zyliu/scratch/tmp",
    dataset_text_field="sft_text",
    optim="adamw_torch",
    learning_rate=lr,
    weight_decay=wd,
    lr_scheduler_type=lr_scheduler_type,
    warmup_ratio=warmup_ratio,
    gradient_accumulation_steps=gradient_accumulation_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=False,
    logging_strategy="steps",
    logging_first_step=True,
    logging_steps=5,
    eval_on_start=True,
    report_to="wandb",
    run_name="sft-dscoder-1B",
    seed=seed,
    max_seq_length=max_sequence_length,
    )

In [ ]:
trainer = SFTTrainer(
    model,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    args=training_args,
    data_collator=collator,
) # type: ignore

trainer.train()
trainer.model.save_pretrained(save_directory=training_args.output_dir)

trainer.accelerator.wait_for_everyone()